# 🧠 Fase de Modelado: Selección Automática de Variables y Regresión
**Proyecto NHANES - Kedro**

El objetivo de este notebook es construir el motor predictivo de Esperanza de Vida para nuestra API de Concientización.
En lugar de seleccionar variables "a mano", utilizaremos un enfoque algorítmico robusto en dos fases:
1. **Feature Selection Automático**: Usaremos Modelos Basados en Árboles (`Random Forest`) y Eliminación Recursiva (`RFECV`) para que la máquina descubra qué hábitos y condiciones de salud impactan realmente en la edad de muerte, descartando variables ruidosas.
2. **Regresión Directa**: Entrenaremos una Regresión Lineal sobre los pacientes fallecidos usando exclusivamente las "variables ganadoras" para predecir el promedio estadístico de supervivencia.


## 1. Carga de Datos desde el Catálogo de Kedro

In [1]:
# Cargar extensión de Kedro (requiere haber iniciado jupyter con 'kedro jupyter lab')
%load_ext kedro.ipython

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Cargar la tabla maestra usando el Catálogo de Kedro
df = catalog.load('model_input')
print(f"Dimensiones de la tabla original: {df.shape}")

[06/09/26 21:10:02] INFO     Using                                                                  ]8;id=162766;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=735880;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packa                
                             ges\kedro\framework\project\rich_logging.yml' as logging                              
                             configuration.                                                                        

[06/09/26 21:10:03] INFO     Registered line magic '%reload_kedro'                                   ]8;id=932658;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=480128;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#64\64]8;;\

                    INFO     Registered line magic '%load_node'                                      ]8;id=509345;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=270078;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#66\66]8;;\

                    INFO     Resolved project path as:                                              ]8;id=571602;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=17650;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#181\181]8;;\
                             C:\Users\alarc\OneDrive\Escritorio\Proyecto_Nhanes_Kedro\proyecto_nhan                
                             es_kedro.                                                                             
                             To set a different path, run '%reload_kedro <project_root>'                           

[06/09/26 21:10:05] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=106967;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=635118;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project Proyecto NHANES Kedro                                    ]8;id=842400;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=885320;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=562584;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=481999;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#148\148]8;;\
                             'pipelines'                                                                           

[06/09/26 21:10:07] INFO     Loading data from model_input (ParquetDataset)...                 ]8;id=443473;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=460561;file://C:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

Dimensiones de la tabla original: (5498, 48)


## 2. Filtrado y Preprocesamiento Inicial

Filtramos para quedarnos únicamente con las personas que fallecieron y preprocesamos las 53 variables disponibles.


In [2]:
# 1. Filtrar solo a los fallecidos
df_dead = df[df['Estado_Mortalidad'] == 1].copy()

# 2. Variable Objetivo: Edad exacta de defunción
df_dead['Edad_Fallecimiento'] = df_dead['Edad'] + (df_dead['Meses_Seguimiento'] / 12.0)

# 3. Remover columnas no predictivas
cols_prohibidas = ['SEQN', 'Estado_Elegibilidad', 'Causa_Muerte', 'Meses_Seguimiento', 'Estado_Mortalidad', 'Edad', 'Edad_Fallecimiento']
X_raw = df_dead.drop(columns=cols_prohibidas)
y = df_dead['Edad_Fallecimiento']

# 4. Preprocesamiento general
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

categorical_cols = ['Genero', 'Etnia', 'Nivel_Educacion', 'Estado_Civil', 'Estado_Diabetes']
numeric_cols = [col for col in X_raw.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ])

X_processed_array = preprocessor.fit_transform(X_raw)

cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = numeric_cols + list(cat_feature_names)

X_processed = pd.DataFrame(X_processed_array, columns=all_feature_names)

print(f"Dataset listo para Feature Selection con {X_processed.shape[1]} predictores.")


Dataset listo para Feature Selection con 53 predictores.


## 3. Selección Automática de Variables (Feature Selection)

Utilizaremos **RFECV (Recursive Feature Elimination with Cross-Validation)** con un Random Forest subyacente. El algoritmo iterará eliminando la variable que menos aporta en cada paso, probando en 5 validaciones cruzadas (5-fold CV) para encontrar el subset que minimice el error de predicción.


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFECV

print("Ejecutando RFECV (Esto puede tomar unos segundos)...")
# Usamos un Random Forest rápido para la selección iterativa
rf_fast = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=5, n_jobs=-1)

# RFECV buscará minimizar el Error Absoluto Medio (MAE)
selector = RFECV(estimator=rf_fast, step=1, cv=5, scoring='neg_mean_absolute_error', min_features_to_select=5)
selector.fit(X_processed, y)

print(f"\n✅ El algoritmo determinó que el número óptimo de features es: {selector.n_features_}")

# Extraer las variables ganadoras
variables_ganadoras = np.array(all_feature_names)[selector.support_]
print("\nLas Variables Ganadoras son:")
for var in variables_ganadoras:
    print(f" - {var}")

# Filtramos nuestro dataset para conservar SOLO las ganadoras
X_final = X_processed[variables_ganadoras]


Ejecutando RFECV (Esto puede tomar unos segundos)...



✅ El algoritmo determinó que el número óptimo de features es: 10

Las Variables Ganadoras son:
 - Ratio_Ingresos_Familiares
 - Promedio_Bebidas_Alcohol
 - IMC
 - Presion_Sistolica
 - Presion_Diastolica
 - Anios_Fumando
 - Glicohemoglobina
 - Colesterol_LDL
 - Creatinina
 - ALT_Enzima_Hepatica


## 4. Entrenamiento de la Regresión Lineal Final

Ahora que la máquina filtró el ruido, entrenamos una regresión limpia e interpretable usando exclusivamente las variables ganadoras.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import pandas as pd

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

# Regresión Lineal Pura con Escalado de Datos (Robusto, sin overfitting)
reg_final = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

reg_final.fit(X_train, y_train)

# Evaluación
y_pred_train = reg_final.predict(X_train)
y_pred_test = reg_final.predict(X_test)

print(f"MAE Train: {mean_absolute_error(y_train, y_pred_train):.2f} años de error promedio")
print(f"MAE Test: {mean_absolute_error(y_test, y_pred_test):.2f} años de error promedio")
print(f"R2 Test: {r2_score(y_test, y_pred_test):.4f}")

print("\n✅ Modelo de Regresión Lineal Robusta entrenado exitosamente sin Overfitting.")

## 5. Simulador de API (What-If)

Probaremos la inferencia simulando a un paciente joven bajo dos escenarios. Almacenamos el modelo final para que el backend de la API pueda consumirlo y generar estas estimaciones dinámicamente.


In [ ]:
# Simulemos perfiles extremos usando cuantiles reales para demostrar el impacto máximo
edad_actual = 25

# Escenario A: Pésimos Hábitos (Percentil 99 de la población)
raw_bad = pd.DataFrame([X_train.quantile(0.99)], columns=variables_ganadoras)
# Para los ingresos, menor es peor
raw_bad['Ratio_Ingresos_Familiares'] = X_train['Ratio_Ingresos_Familiares'].quantile(0.01)

# Escenario B: Excelentes Hábitos (Percentil 1 de la población)
raw_good = pd.DataFrame([X_train.quantile(0.01)], columns=variables_ganadoras)
raw_good['Ratio_Ingresos_Familiares'] = X_train['Ratio_Ingresos_Familiares'].quantile(0.99)

what_if_df = pd.concat([raw_bad, raw_good], ignore_index=True)

esperanzas = reg_final.predict(what_if_df)

esperanza_a = int(max(esperanzas[0], edad_actual))
esperanza_b = int(max(esperanzas[1], edad_actual))
ganancia = esperanza_b - esperanza_a

print(f"=== REPORTE DE CONCIENTIZACIÓN (Paciente simulado de {edad_actual} años) ===")
print(f"Escenario A (Pésimos hábitos, percentil 99): Vivirá hasta los {esperanza_a} años.")
print(f"Escenario B (Excelentes hábitos, percentil 1): Vivirá hasta los {esperanza_b} años.")
print(f"\n¡Puedes GANAR {ganancia} AÑOS DE VIDA si cambias tus hábitos HOY!")